In [ ]:
#!pip install pymupdf --q

In [ ]:
# pip uninstall -y langchain langchain-core langchain-community langchain-openai langsmith


In [ ]:
# pip install langchain==0.2.17 langchain-core==0.2.43 langchain-community==0.2.19 langchain-openai==0.1.25 langsmith==0.1.147

In [2]:
import os
import langchain, langchain_community
print(langchain.__version__)
print(langchain_community.__version__)

0.2.17
0.2.19


In [3]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
import utils

In [4]:
HR_DOC_NAME = "ACME Corp Employee Leave Policy Handbook.pdf"
PROC_DOC_NAME = "Acme_Corp_IT_Procurement_Policy.pdf"
VECTOR_DB_PATH = "local_faiss_db"

In [5]:
hr_docs = PyMuPDFLoader(HR_DOC_NAME).load()

In [6]:
hr_metadata = {"department": "HR"}
proc_metadata = {"department": "PROC"}

In [7]:
for doc in hr_docs:
    doc.metadata.update(hr_metadata)

In [8]:
proc_docs = PyMuPDFLoader(PROC_DOC_NAME).load()

In [9]:
for doc in proc_docs:
    doc.metadata.update(proc_metadata)

In [10]:
final_doc = proc_docs + hr_docs

In [11]:
final_doc[0].metadata

{'source': 'Acme_Corp_IT_Procurement_Policy.pdf',
 'file_path': 'Acme_Corp_IT_Procurement_Policy.pdf',
 'page': 0,
 'total_pages': 3,
 'format': 'PDF 1.7',
 'title': 'Acme Corp - IT Procurement Policy',
 'author': '',
 'subject': '',
 'keywords': '',
 'creator': '',
 'producer': 'WeasyPrint 62.3',
 'creationDate': '',
 'modDate': '',
 'trapped': '',
 'department': 'PROC'}

In [12]:
text_splitter  = RecursiveCharacterTextSplitter(
    chunk_size=600,
    chunk_overlap=50,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""]
)
splitted_text=text_splitter.split_documents(final_doc)

In [13]:
len(splitted_text)

19

In [14]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

In [15]:
vectordb = FAISS.from_documents(
    documents=splitted_text,
    embedding=embeddings
)

In [16]:
retriever = vectordb.as_retriever(search_kwargs={"k": 2, "filter": {"department": "HR"}})

In [17]:
print(retriever.invoke("What is the policy for taking a vacation leave")[0].page_content)

note that a formal medical certificate must be submitted to HR for any sick leave absences 
exceeding three consecutive days. 
2. Vacation Leave 
Full-time employees become eligible to take accrued vacation leave after completing their initial 
ninety-day probationary period. The company provides fifteen days of paid vacation leave per 
calendar year. Unused vacation days up to a maximum of five days can be carried over to the 
next calendar year, after which any remaining unused balance is forfeited. Vacation leave must


In [18]:
from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain.chains import RetrievalQA

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [19]:
@tool
def get_procurement_response(query: str) -> str:
    """ This tool will take input and retrieve response from Vector Databases for Procurement related questions """
    retriever = vectordb.as_retriever(search_kwargs={"k": 2, "filter": {"department": "PROC"}})
    Retriever_chain = RetrievalQA.from_chain_type(llm,
                                              retriever=retriever,
                                              return_source_documents=True
                                             )
    response = Retriever_chain.invoke(query)
    return response.get('result')

@tool
def get_hr_response(query: str) -> str:
    """ This tool will take input and retrieve response from Vector Databases for Human Resources (HR) related questions """
    retriever = vectordb.as_retriever(search_kwargs={"k": 2, "filter": {"department": "HR"}})
    Retriever_chain = RetrievalQA.from_chain_type(llm,
                                              retriever=retriever,
                                              return_source_documents=True
                                             )
    response = Retriever_chain.invoke(query)
    return response.get('result')

In [20]:
SYSTEM_PROMPT = """You are helpful agent who can answer question on Procurement and Human Resources domains.
You should use the tool given to respond to the question.
Do not use any other knowledge for the response
"""

In [21]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system", SYSTEM_PROMPT
    ),
    (
        "human", "User: {input}"
    ),
    MessagesPlaceholder(
        variable_name="agent_scratchpad"
    )
])

In [22]:
tools = [get_hr_response, get_procurement_response]

In [23]:
agent = create_tool_calling_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [24]:
executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True    
)

In [25]:
response = executor.invoke({"input": "How many days of sick leave do I get?"})
print("\n--- Agent Output ---")
print(response["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `get_hr_response` with `{'query': 'How many days of sick leave do I get?'}`


You receive twelve days of paid sick leave per calendar year.You receive twelve days of paid sick leave per calendar year.

> Finished chain.

--- Agent Output ---
You receive twelve days of paid sick leave per calendar year.


In [26]:
response = executor.invoke({"input": "Who are the reviewers and approvers for purchase under $5000 purchase?"})
print("\n--- Agent Output ---")
print(response["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `get_procurement_response` with `{'query': 'Who are the reviewers and approvers for purchases under $5000?'}`


For purchases under $5,000, the required reviewers are the IT Infrastructure Team, and the final approver is the Department Head.For purchases under $5,000, the required reviewers are the IT Infrastructure Team, and the final approver is the Department Head.

> Finished chain.

--- Agent Output ---
For purchases under $5,000, the required reviewers are the IT Infrastructure Team, and the final approver is the Department Head.


In [ ]:
#

In [27]:
response = executor.invoke({"input": "Before entering into formal negotiations or executing contracts with technology vendors, what are the due diligence steps? Summarize your answer in less than 100 words"})
print("\n--- Agent Output ---")
print(response["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `get_procurement_response` with `{'query': 'What are the due diligence steps before entering into formal negotiations or executing contracts with technology vendors?'}`


The due diligence steps before entering into formal negotiations or executing contracts with technology vendors include:

1. **Architecture and Compatibility Assessment**: The IT Architecture Board must evaluate the proposed solution to ensure seamless integration with current core systems. Precedence must be given to standard enterprise solutions already deployed within the organization before introducing novel platforms.

2. **Information Security & Privacy Evaluation**: This step involves assessing the vendor's compliance with information security and privacy standards. 

These assessments are crucial to ensure that the selected technology vendor aligns with the organization's existing systems and security requirements.Before entering into formal negotiations or executing contracts with technology vendor

In [28]:
response = executor.invoke({"input": "When do I become eligible for vacation leave"})
print("\n--- Agent Output ---")
print(response["output"])

Error in StdOutCallbackHandler.on_chain_start callback: AttributeError("'NoneType' object has no attribute 'get'")



Invoking: `get_hr_response` with `{'query': 'When do I become eligible for vacation leave?'}`


You become eligible to take accrued vacation leave after completing your initial ninety-day probationary period.You become eligible to take accrued vacation leave after completing your initial ninety-day probationary period.

> Finished chain.

--- Agent Output ---
You become eligible to take accrued vacation leave after completing your initial ninety-day probationary period.
